# ViT image-similarity experiment

Historical cat/dog/query toy example from the USC project notebook. It is an image-embedding experiment, not clinical image retrieval.

Place three shareable images at `examples/images/cat.jpg`, `dog.jpg`, and `query.jpg` relative to the notebook working directory. Supply your own images; none are included. Running the example downloads Google's pretrained ViT weights.


In [ ]:
import torch
import torch.nn.functional as F
from transformers import ViTModel, ViTImageProcessor
from PIL import Image
import matplotlib.pyplot as plt


In [ ]:
model = ViTModel.from_pretrained('google/vit-base-patch16-224-in21k')
model.eval()
imgpathcat="examples/images/cat.jpg"
imgpathdog="examples/images/dog.jpg"
# Load the image processor for preprocessing the images
processor = ViTImageProcessor.from_pretrained('google/vit-base-patch16-224-in21k')
# Load an image (example)
imagecat = Image.open(imgpathcat).convert("RGB")
imagedog = Image.open(imgpathdog).convert("RGB")



# Preprocess the image
cat_inputs = processor(images=imagecat, return_tensors="pt")
dog_inputs = processor(images=imagedog, return_tensors="pt")

# Generate embeddings for the cat and dog images
with torch.no_grad():
    cat_embedding = model(**cat_inputs).last_hidden_state[:, 0, :]
    dog_embedding = model(**dog_inputs).last_hidden_state[:, 0, :]

def cosine_similarity(a, b):
    return F.cosine_similarity(a, b, dim=-1)


test_image = Image.open('examples/images/query.jpg').convert('RGB')
new_inputs = processor(images=test_image, return_tensors="pt")

# Generate embedding for the new image
with torch.no_grad():
    new_embedding = model(**new_inputs).last_hidden_state[:, 0, :]

# Calculate similarity to cat and dog embeddings
similarity_to_cat = cosine_similarity(new_embedding, cat_embedding)
similarity_to_dog = cosine_similarity(new_embedding, dog_embedding)

# Determine which one is more similar
if similarity_to_cat > similarity_to_dog:
    print("The image is more similar to the cat.")
else:
    print("The image is more similar to the dog.")
# print(cat_embedding)
fig, axs = plt.subplots(1, 3, figsize=(15, 5))

# Display the cat image
axs[0].imshow(imagecat)
axs[0].set_title('Cat Image')
axs[0].axis('off')

# Display the dog image
axs[1].imshow(imagedog)
axs[1].set_title('Dog Image')
axs[1].axis('off')

# Display the test image
axs[2].imshow(test_image)
axs[2].set_title('Test Image')
axs[2].axis('off')

# Show the plot
plt.show()
